# Sendov's Conjecture

In [ ]:
#@title Verification code

# pylint: disable=unused-import
# pylint: disable=g-bad-import-order




def ideal_x_n_minus_1_coeffs(n):
  """Returns the coefficients of the polynomial x^n - 1."""
  coeffs = np.zeros(n + 1)
  coeffs[0] = 1  # Coefficient of x^n
  coeffs[-1] = -1  # Constant term
  return coeffs


def coefficient_distance_to_x_n_minus_c_family(roots):
  """Calculates a penalty based on how close the polynomial's coefficients are to the form x^n - c (family of polynomials).
  """
  coeffs = polynomial_roots_to_coeffs(roots)
  degree = len(coeffs) - 1

  # Initialize penalty (we'll sum squared magnitudes of intermediate
  # coefficients)
  penalty_distance = 0

  # Iterate through intermediate coefficients (from x^(n-1) down to x^1)
  for i in range(1, degree):  # Indices 1 to n-1 correspond to x^(n-1) to x^1
    coeff_index = i  # Index in the coefficient array (coeffs)
    coeff_magnitude_squared = np.abs(coeffs[coeff_index]) ** 2
    penalty_distance += coeff_magnitude_squared

  # You could also penalize deviation of the leading coefficient from 1
  # (optional):
  leading_coeff_penalty = np.abs(coeffs[0] - 1) ** 2
  penalty_distance += leading_coeff_penalty

  return penalty_distance


def polynomial_roots_to_coeffs(roots):
  """Converts roots of a polynomial to its coefficients."""
  return np.poly(roots)


def polynomial_coeffs_to_derivative_coeffs(coeffs):
  """Calculates coefficients of the derivative of a polynomial."""
  derivative_coeffs = np.polyder(coeffs)
  return derivative_coeffs


def polynomial_critical_points(roots):
  """Calculates critical points of a polynomial given its roots."""
  coeffs = polynomial_roots_to_coeffs(roots)
  derivative_coeffs = polynomial_coeffs_to_derivative_coeffs(coeffs)
  critical_points = np.roots(derivative_coeffs)
  return critical_points


def calculate_extremal_family_penalty(
    roots, critical_points, max_min_distance, tolerance=0.75
):
  """Calculates penalty if construction resembles the extremal family."""
  penalty = 0.0
  special_zero_found = False

  for i in range(len(roots)):
    special_zero = roots[i]

    # Calculate distances from the special zero to all critical points
    distances_to_special_zero = np.abs(critical_points - special_zero)

    # Check if all critical points are approximately at max_min_distance from
    # the special zero
    if np.all(np.abs(distances_to_special_zero - max_min_distance) < tolerance):
      special_zero_found = True
      break  # No need to check other roots if one special zero is found

  if special_zero_found:
    penalty = 1.0  # Increased penalty for closer resemblance to extremal family

  return penalty


def evaluate_sendov_conjecture(roots):
  """Evaluates Sendov's conjecture for a given set of roots (optimized).

  Args:
    roots: A numpy array of complex numbers representing the roots of a
      polynomial.

  Returns:
    The maximum distance from each root to its closest critical point,
    with a penalty for certain polynomial forms.
  """
  n = len(roots)
  if n < 15:
    return 0  # Conjecture is for degree n >= 2

  # Project roots onto the unit circle (if outside)
  for i in range(len(roots)):
    if np.abs(roots[i]) > 1:
      roots[i] = roots[i] / np.abs(roots[i])  # Project to unit circle

  critical_points = polynomial_critical_points(roots)

  # Vectorized distance calculation and minimum finding
  distances = np.abs(roots[:, np.newaxis] - critical_points)  # shape (n, n-1)
  min_distances = np.min(distances, axis=1)  # shape (n,)
  max_min_distance = np.max(min_distances)

  penalty = calculate_extremal_family_penalty(
      roots, critical_points, max_min_distance
  )

  coeff_unity_distance = coefficient_distance_to_x_n_minus_c_family(roots)
  if coeff_unity_distance < n:
    penalty += 1.0
  else:
    penalty += 0

  return max_min_distance - penalty


############### #Hidden variants of above code


def polynomial_roots_to_coeffs_h(roots):
  """Converts roots of a polynomial to its coefficients."""
  return np.poly(roots)


def polynomial_coeffs_to_derivative_coeffs_h(coeffs):
  """Calculates coefficients of the derivative of a polynomial."""
  derivative_coeffs = np.polyder(coeffs)
  return derivative_coeffs


def polynomial_critical_points_h(roots):
  """Calculates critical points of a polynomial given its roots."""
  coeffs = polynomial_roots_to_coeffs_h(roots)
  derivative_coeffs = polynomial_coeffs_to_derivative_coeffs_h(coeffs)
  critical_points = np.roots(derivative_coeffs)
  return critical_points


def evaluate_sendov_conjecture_h(roots: np.ndarray):
  """Evaluates Sendov's conjecture for a given set of roots (optimized).

  Args:
    roots: A numpy array of complex numbers representing the roots of a
      polynomial.

  Returns:
    The maximum distance from each root to its closest critical point,
    with a penalty for certain polynomial forms.
  """
  n = len(roots)
  if n < 2:
    return 0  # Conjecture is for degree n >= 2

  # Project roots onto the unit circle (if outside)
  for i in range(len(roots)):
    if np.abs(roots[i]) > 1:
      roots[i] = roots[i] / np.abs(roots[i])  # Project to unit circle

  critical_points = polynomial_critical_points_h(roots)

  # Vectorized distance calculation and minimum finding
  distances = np.abs(roots[:, np.newaxis] - critical_points)  # shape (n, n-1)
  min_distances = np.min(distances, axis=1)  # shape (n,)
  max_min_distance = np.max(min_distances)

  penalty = calculate_extremal_family_penalty(
      roots, critical_points, max_min_distance
  )
  coeff_unity_distance = coefficient_distance_to_x_n_minus_c_family(roots)
  if coeff_unity_distance < n:
    penalty += 1.0
  else:
    penalty += 0

  return max_min_distance - penalty


def format_feedback_repr(feedback):
  """Formats feedback dictionary for representation."""
  formatted_feedback = {}
  for key, value in feedback.items():
    if isinstance(value, np.ndarray):
      repr_str = repr(value)  # Get repr string (e.g., "array([[...], [...]])")
      cleaned_repr_str = re.sub(r'[\n\s]+', ' ', repr_str)  # Clean up

      # Remove the leading "array(" and trailing ")" from repr string, then wrap
      # with "np.array(...)"
      array_content = cleaned_repr_str[
          6:-1
      ]  # Extract content inside "array(...)"

      if np.iscomplexobj(value):
        formatted_feedback[key] = (  # Use extracted content in np.array
            f'np.array({array_content}, dtype=np.complex128)'
        )
      elif not np.issubdtype(value.dtype, np.inexact):
        formatted_feedback[key] = f'np.array({array_content}, dtype=np.float64)'
      else:
        formatted_feedback[key] = f'np.array({array_content})'

    else:
      formatted_feedback[key] = repr(value)
  return formatted_feedback


def evaluate(
    hypers: Mapping[str, Any],
) -> tuple[dict[str, float], dict[str, str]]:
  """Returns the numerical bound for the polygons if valid, or 0 if invalid."""
  result = {}
  feedback = {}
  del hypers
  best_construction = search_for_best_poly()
  result['score'] = evaluate_sendov_conjecture_h(best_construction)

  feedback['best_roots'] = best_construction
  feedback['best_score_found'] = result['score']
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Data and verification

# Best construction found by AlphaEvolve
# These are the roots of a degree-26 polynomial that achieves a high
# max-min distance to critical points.
best_roots = np.array([
    0.99693524+0.07824373j, 0.97498211+0.22218974j, 0.90700938+0.42102798j,
    0.78801684+0.61569098j, 0.62778926+0.77832891j, 0.43510627+0.90038069j,
    0.22370288+0.97464138j, 0.00632288+0.99998001j, -0.21044908+0.97762002j,
    -0.41760805+0.90860089j, -0.60540285+0.79597424j, -0.76437044+0.64486088j,
    -0.88700614+0.46172072j, -0.96799266+0.25096914j, -0.99919207+0.04019379j,
    -0.97735424-0.21157095j, -0.90263735-0.43040379j, -0.77992424-0.62583862j,
    -0.61558987-0.78809525j, -0.41795973-0.90843897j, -0.20038765-0.97722498j,
     0.02538627-0.99967776j,  0.25105268-0.96797103j,  0.46645091-0.88454175j,
     0.66056254-0.75078929j,  0.83046791-0.55705756j
])

score = evaluate_sendov_conjecture_no_penalty(best_roots)
print(f"Number of roots: {len(best_roots)}")
print(f"Max min-distance to critical points: {score:.6f}")
print(f"All roots inside unit disk: {np.all(np.abs(best_roots) <= 1.0 + 1e-10)}")

In [ ]:
#@title Initial program

"""Finds a function that give the best bound for Sendov's conjecture."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import re
from typing import Any, Callable, Mapping
import scipy.linalg as la
import numpy.polynomial.polynomial as poly

minimize = optimize.minimize



def search_for_best_poly() -> np.ndarray:
  """Function to search for the best construction."""
  roots = np.array([
      0.1 + 0.1j,
      -0.2 - 0.3j,
      0.4,
      -0.5,
      0.3,
      0.1,
      -1,
      0.8 + 0.1j,
      0.9 - 0.1j,
      -0.8j,
      0.7,
      0.1 + 0.2j,
      0.2 - 0.1j,
      0.3 + 0.1j,
      0.4 - 0.1j,
      0.5 - 0.1j,
      0.6 - 0.1j,
      0.7 - 0.1j,
      0.8 - 0.16j,
      0.9 - 0.19j,
      0.543 - 0.1432j,
      0.343 - 0.1234j,
      0.43 - 0.6j,
      0.3 - 0.1j,
      0.6 - 0.1j,
      0.234 - 0.62j,
  ])
  best_roots = roots.copy()
  best_score = evaluate_sendov_conjecture(roots)
  start_time = time.time()
  while time.time() - start_time < np.random.randint(10, 500):
    roots[np.random.randint(2)] += np.random.uniform(-0.1, 0.1)
    score = evaluate_sendov_conjecture(roots)
    if score > best_score:
      best_score = score
      best_roots = roots.copy()
      print(score)
  return best_roots

**Prompt used**

Act as an expert software developer and inequality specialist specializing in
creating polynomials with certain properties. You will be trying to find
counterexamples to Sendov's conjecture, which states that for a polynomial with
all roots $r_1, ..., r_n$ inside the closed unit disk $|z| \leq 1$, each of the $n$ roots is at a distance no more than 1 from at least one critical point.

Your task is to write a search function that searches for the best list of roots.
Your function will have 500 seconds to run, and after that it has to have returned
the best construction it found. If after 500 seconds it has not returned anything,
it will be terminated with negative infinity points. You may freely choose the value
of n, the number of zeros. To find a counterexample where the final score is greater
than 1, you will likely have to look in the $n > 20$ range. Your $n$ must be at least 15
for it to receive a score.

You may code up any search method you want, and you are allowed to call the
evaluate_sendov_conjecture() function as many times as you want.
You have access to it, you don't need to code up the evaluate_sendov_conjecture() function.

Polynomials of the form $x^n - 1$ (where the roots are the $n$-th roots of unity)
are known to be local optima. You will get a penalty if you get close to them, to help you not get stuck in them.


## What AlphaEvolve found

AlphaEvolve found the expected maximizers $p(z) = c(z^n - e^{i\theta})$ for $c \neq 0$ and real $\theta$, as well as near-maximizers such as $p(z) = z^n - z$, but did not discover any additional maximizers beyond these known examples.